In [12]:

%%writefile cuda.cu
#include <iostream>
#include <cuda_runtime.h>
int getCoresPerSM(int major, int minor) {
    // Defines cores per SM based on architecture generation
    switch (major) {
 case 2: // Fermi
            return (minor == 1) ? 48 : 32;
        case 3: // Kepler
            return 192;
        case 5: // Maxwell
            return 128;
        case 6: // Pascal
            if (minor == 1 || minor == 2) return 128;
            if (minor == 0) return 64;
            return 128; // Default fallback for Pascal
        case 7: // Volta (7.0), Turing (7.5)
            return 64;
        case 8: // Ampere (8.0, 8.6, 8.7), Ada Lovelace (8.9)
            if (minor == 0) return 64;
            if (minor == 6 || minor == 9) return 128;
            return 64; // Default fallback for Ampere variants
        case 9: // Hopper (9.0), Blackwell (9.5)
            return 128;
        default:
            return 128; // Standard fallback for future architectures
    }
}
int main() {
    int deviceCount = 0;

    // Get the total number of CUDA-enabled devices
    cudaError_t error = cudaGetDeviceCount(&deviceCount);

    if (error != cudaSuccess) {
        std::cerr << "CUDA Error: " << cudaGetErrorString(error) << std::endl;
        return 1;
    }

    std::cout << "Found " << deviceCount << " CUDA device(s).\n" << std::endl;

    // Loop through each available device
    for (int i = 0; i < deviceCount; ++i) {
        cudaDeviceProp prop;

        // Populate the property structure for the current device index
        cudaGetDeviceProperties(&prop, i);

        std::cout << "--- Device " << i << ": " << prop.name << " ---" << std::endl;
        std::cout << "  Compute Capability:          " << prop.major << "." << prop.minor << std::endl;
        std::cout << "  Total Global Memory:         " << prop.totalGlobalMem / (1024 * 1024) << " MB" << std::endl;
        std::cout << "  Streaming Multiprocessors:   " << prop.multiProcessorCount << std::endl;
        std::cout << "  Cores Per SM:                " << getCoresPerSM(prop.major, prop.minor) << std::endl;
        std::cout << "  Total Cores:                 " << prop.multiProcessorCount*getCoresPerSM(prop.major, prop.minor) << std::endl;
        std::cout << "  Max Threads Per Block:       " << prop.maxThreadsPerBlock << std::endl;
        std::cout << "  Shared Memory Per Block :     " << prop.sharedMemPerBlock / 1024 << " KB" << std::endl;
        std::cout << "  Warp Size:                   " << prop.warpSize << std::endl;
                std::cout << "Clock Rate: "
          << prop.clockRate / 1000
          << " MHz" << std::endl;

        std::cout << "Memory Clock Rate: "
                  << prop.memoryClockRate / 1000
                  << " MHz" << std::endl;

        std::cout << "Memory Bus Width: "
                  << prop.memoryBusWidth
                  << " bits" << std::endl;

        std::cout << "L2 Cache Size: "
                  << prop.l2CacheSize / 1024
                  << " KB" << std::endl;

        std::cout << "Registers Per Block: "
                  << prop.regsPerBlock
                  << std::endl;

        std::cout << "Max Threads Per SM: "
                  << prop.maxThreadsPerMultiProcessor
                  << std::endl;

        std::cout << "Max Grid Size: "
                  << prop.maxGridSize[0] << ", "
                  << prop.maxGridSize[1] << ", "
                  << prop.maxGridSize[2]
                  << std::endl;

        std::cout << "Max Block Dimensions: "
                  << prop.maxThreadsDim[0] << " x "
                  << prop.maxThreadsDim[1] << " x "
                  << prop.maxThreadsDim[2]
                  << std::endl;
        std::cout << std::endl;
    }

    return 0;
}



Overwriting cuda.cu


In [13]:
!nvcc cuda.cu -o cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [14]:
!./cuda

Found 1 CUDA device(s).

--- Device 0: Tesla T4 ---
  Compute Capability:          7.5
  Total Global Memory:         14912 MB
  Streaming Multiprocessors:   40
  Cores Per SM:                64
  Total Cores:                 2560
  Max Threads Per Block:       1024
  Shared Memory Per Block :     48 KB
  Warp Size:                   32
Clock Rate: 1590 MHz
Memory Clock Rate: 5001 MHz
Memory Bus Width: 256 bits
L2 Cache Size: 4096 KB
Registers Per Block: 65536
Max Threads Per SM: 1024
Max Grid Size: 2147483647, 65535, 65535
Max Block Dimensions: 1024 x 1024 x 64



In [15]:

%%writefile barrier.cu
#include <stdio.h>

__device__ int count = 0;

__global__ void barrierKernel()
{
    int threadid = threadIdx.x;

    printf("Thread %d reached barrier\n", threadid);

    atomicAdd(&count, 1);

    while(count < blockDim.x);

    __syncthreads();

    printf("Thread %d passed barrier\n", threadid);
}

int main()
{
    barrierKernel<<<1,10>>>();

    cudaDeviceSynchronize();

    return 0;
}

Overwriting barrier.cu


In [16]:
!nvcc barrier.cu -o barrier

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [17]:
!./barrier

Thread 0 reached barrier
Thread 1 reached barrier
Thread 2 reached barrier
Thread 3 reached barrier
Thread 4 reached barrier
Thread 5 reached barrier
Thread 6 reached barrier
Thread 7 reached barrier
Thread 8 reached barrier
Thread 9 reached barrier
Thread 0 passed barrier
Thread 1 passed barrier
Thread 2 passed barrier
Thread 3 passed barrier
Thread 4 passed barrier
Thread 5 passed barrier
Thread 6 passed barrier
Thread 7 passed barrier
Thread 8 passed barrier
Thread 9 passed barrier
